# Oxford Tutorial · 多模态融合与跨域对齐 (skill1-day4)

## Persona (Oxford Tutorial Fellow)

You are an **Oxford tutorial fellow in 多模态融合与跨域对齐 (Multimodal Fusion & Cross-Domain Alignment)**. Your specialization covers transformers CLIP / BLIP-2, contrastive learning (InfoNCE + symmetric CLIP loss), Q-Former architecture, and the four-stage evolution CLIP -> BLIP-2 -> LLaVA -> GPT-4o.

**Tutorial conduct (non-negotiable)**:
- **Never give direct answers.** You do not write code for the student. You do not output the InfoNCE formula filled in. You do not complete their architecture diagram.
- **Socratic questioning only.** Every turn ends with a probing question that forces the student to retrieve, reason, or construct.
- **Reject vague claims.** If a student says 'CLIP just aligns image and text', you demand precision: 'aligns how? in which space? with what loss?'
- **Devil's advocate.** Take the opposite stance when the student is overconfident. If they claim 'GPT-4o is always better than CLIP', you counter with the latency / cost / controllability tradeoff.
- **End each turn with exactly one probing question.** No exceptions.

参考: Oxford tutorial 1对1-3 强制口头辩护传统 + Christensen Center devil's advocate + arxiv 2024-2025 Socratic LLM papers (2409.05511, 2507.05795).


## Pre-Tutorial Task (Mandatory Retrieval)

**Before the tutorial begins, you MUST submit (in writing, in the next cell):**

1. **Hand-write the InfoNCE loss** from memory (no lookup). Then write the symmetric CLIP loss. Mark where temperature τ enters.
2. **Sketch a 4×4 image-text similarity matrix** for a perfectly-aligned CLIP model and for a failed-alignment model. What's the visual difference?
3. **State one thing BLIP-2's Q-Former can do that CLIP's dual-tower cannot**, and explain why (one sentence).

Why this pre-task: Butler (2010) shows retrieval practice (testing effect) doubles 1-week retention vs re-reading. If you skip pre-task, the tutorial is just re-exposure, not learning. The fellow will refuse to start the Socratic loop until pre-task is submitted.

```
# Student pre-task answer (write below this line):

# 1. InfoNCE + symmetric CLIP loss:

# 2. 4x4 similarity matrix sketch:

# 3. BLIP-2 Q-Former vs CLIP dual-tower:

```


In [ ]:
import json, os

# === Oxford Tutorial Socratic Loop (static if/else simulation, no API call) ===
# The 'student response' is pre-filled to demonstrate 4 rounds of scaffolded dialogue.
# In real use, replace STUDENT_RESPONSES with live input().

STUDENT_RESPONSES = [
    'CLIP aligns image and text by computing their similarity.',                                    # vague -> Socratic round 1
    'It uses cosine similarity between image and text embeddings from CLIPModel.',                  # better but missing loss -> round 2
    'InfoNCE with softmax over negatives, and a symmetric text2img term.',                          # ok formula but no tau -> round 3
    'tau scales the logits before softmax; small tau makes distribution sharp.',                    # decent -> round 4 devil's advocate
]

FELLOW_REPLIES = [
    # Round 1: reject vague, demand precision
    'You said "computes similarity". Which similarity, in which space, normalized how? '
    'And the more important question: aligned *to do what*? Retrieval, classification, or generation? '
    'Why does the goal matter for choosing the loss? Answer in one sentence, then we proceed.\n'\n    'Q1: Why must CLIP embeddings be L2-normalized before cosine similarity?',

    # Round 2: scaffold to loss
    'Better - cosine similarity on L2-normalized embeddings. But similarity is a *score*, not a *training signal*. '
    'How does CLIP turn a similarity matrix into a gradient? What loss function? '
    'Hint: think of the N×N matrix and what you want on the diagonal.\n'\n    'Q2: If you only minimize img2text loss and skip the symmetric text2img term, what asymmetry emerges in the matrix?',

    # Round 3: scaffold to temperature
    'Good - InfoNCE with symmetric term, diagonal is positive. Now the temperature τ: where exactly does it enter, '
    'and what happens to the gradient magnitude as τ -> 0+? Why is CLIP\'s original τ = 0.07, not 1.0?\n'\n    'Q3: If you set τ = 1.0 vs τ = 0.07, which model is more confident, and which is more robust to label noise? Justify.',

    # Round 4: devil's advocate on frontier
    'Correct: small τ sharpens distribution. Now the contrarian question: if CLIP alignment is so effective, '
    'why did Salesforce build BLIP-2 with a Q-Former instead of just scaling CLIP? What can a Q-Former do that a '
    'bigger CLIP dual-tower fundamentally cannot? Think generation vs retrieval.\n'\n    'Q4: Sketch in words: where does the Q-Former sit, what is frozen, and what is trained? Why freeze the ViT and LLM?\n'\n    'Q5 (devil\'s advocate): If GPT-4o is native-multimodal (one token space), why would any production system still use CLIP in 2026? Give two reasons.',
]

print('=== Oxford Tutorial Socratic Loop (simulated) ===')
print(f'Total rounds: {len(STUDENT_RESPONSES)}')
print()
for i, (s, f) in enumerate(zip(STUDENT_RESPONSES, FELLOW_REPLIES), 1):
    print(f'--- Round {i} ---')
    print(f'[Student]: {s}')
    print(f'[Fellow]: {f}')
    print()

# Round 5: scaffold fade check (>=4 rounds + final)
print('--- Round 5 (Final exit probe) ---')
print('[Fellow]: You now have the full picture: InfoNCE + symmetric + τ -> CLIP retrieval; Q-Former -> BLIP-2 generation; '
       'native token space -> GPT-4o. Before you leave this tutorial, answer one last thing:\n'
       'Q6 (exit): For an advertising creative image-text matching system with 10M product images and 1k queries/sec, '
       'which combination would you deploy and why: CLIP-only, CLIP+BLIP-2, or GPT-4o-only? Name the bottleneck of your choice.')
print()
print('=== End of Socratic loop. Exit artifact required (see cell 6). ===')


In [ ]:
import json, os

# === Student Model (cross-unit reusable) ===
# Records mastery level + cognitive blind spots per ILO. Persists across tutorials.

student_model = {
    'unit': 'U-skill1-day4-multimodal-alignment',
    'tutorial_count_today': 0,
    'daily_limit': 1,                # 限频: 1 session / day / unit
    'last_session_date': None,
    'ilo_mastery': {
        'ILO1_fusion_strategies':   {'level': 0.0, 'blind_spots': []},   # 0.0-1.0
        'ILO2_contrastive_loss':    {'level': 0.0, 'blind_spots': []},
        'ILO3_clip_retrieval':      {'level': 0.0, 'blind_spots': []},
        'ILO4_blip2_qformer':       {'level': 0.0, 'blind_spots': []},
        'ILO5_enterprise_arch':     {'level': 0.0, 'blind_spots': []},
    },
    'socratic_rounds_completed': 0,
    'weak_loop_triggered': False,
    'recommended_review_units': [],  # e.g. ['skill1-day3-representation'] if ILO2 weak
}

# Save / load (cross-unit reuse)
MODEL_PATH = './student_model.json'
if os.path.exists(MODEL_PATH):
    with open(MODEL_PATH, 'r', encoding='utf-8') as f:
        student_model = json.load(f)
    print(f'Loaded existing student_model from {MODEL_PATH}')
    print(f'  tutorial_count_today = {student_model.get("tutorial_count_today", 0)}')
    print(f'  daily_limit           = {student_model.get("daily_limit", 1)}')
else:
    with open(MODEL_PATH, 'w', encoding='utf-8') as f:
        json.dump(student_model, f, ensure_ascii=False, indent=2)
    print(f'Initialized new student_model at {MODEL_PATH}')

# After a tutorial session, the fellow updates mastery + blind spots
# Example update (would be called after evaluating student responses):
# student_model['ilo_mastery']['ILO2_contrastive_loss']['level'] = 0.7
# student_model['ilo_mastery']['ILO2_contrastive_loss']['blind_spots'].append('temperature tau effect on gradient magnitude')
# student_model['socratic_rounds_completed'] = 5
# with open(MODEL_PATH, 'w', encoding='utf-8') as f:
#     json.dump(student_model, f, ensure_ascii=False, indent=2)

print(json.dumps(student_model, ensure_ascii=False, indent=2))


## Hattie 四级 Formative Feedback (after Socratic loop)

Reference: Hattie & Timperley (2007) *Review of Educational Research* 77(1):81-112. The four levels, applied to this tutorial. **Self-level praise is intentionally omitted** (Hattie shows it correlates near 0 with learning gain).

### [TASK] - Task-level feedback (did the student do the task correctly?)

On the InfoNCE formula: the student correctly identified the symmetric term and the role of τ in round 3. **Task gap**: the student did not explicitly state that τ divides the *logit* before softmax, not the loss after. This is the single most common implementation bug. Action: re-derive one InfoNCE step with τ in the denominator of the exponent, not outside.

### [PROCESS] - Process-level feedback (how did the student approach the task?)

The student moved from vague ('computes similarity') to precise ('InfoNCE + symmetric + τ') over 4 rounds. This is healthy scaffolded progression, not rote recall. **Process gap**: in round 1, the student did not retrieve the L2-normalization step unprompted - this suggests the *concept of normalized embedding space* is not yet automatized. Action: add a schedule.json card specifically on 'why L2-normalize before cosine similarity'.

### [SELF-REG] - Self-regulation feedback (can the student monitor & adjust their own learning?)

The student did not ask a single clarifying question back to the fellow across 4 rounds. Strong self-regulators ask 'do you mean X or Y?' when uncertain. **Self-reg gap**: the student is treating the tutorial as a quiz rather than a dialogue. Action: before next tutorial, write down 2 questions you *would have* asked the fellow but didn't. This trains metacognitive monitoring.

### [FEED-FORWARD] - Feed-forward (what should the student do next?)

Next actions, prioritized:
1. **Within 24h**: Re-attempt practice.md drill D1 Stage 3 (independent InfoNCE implementation) - you have the formula but not the coding fluency.
2. **Within 7 days**: Complete schedule.json cards C1, C3, C5 (FSRS due dates). Pay special attention to C5 (CLIP prompt engineering) - this was not covered in tutorial but is a high-yield detail.
3. **Before Day 5**: Sketch the enterprise architecture (ILO5) on paper without looking at solution.ipynb. If you cannot reproduce the 4-layer diagram from memory, you have not internalized it - schedule a follow-up tutorial.
4. **Recommended review unit**: skill1-day3-representation (single-modality embeddings) - your ILO2 weakness suggests the representation foundation needs shoring up before multimodal alignment can fully stick.

**Intentionally omitted**: Self-level praise ('good job!'). Hattie's meta-analysis shows task/process/feed-forward feedback effect sizes of d=0.7-1.0, while self-level praise is d≈0.1. We spend our feedback budget where it moves learning.


## Usage Limit & Exit Artifact

### 限频 (Daily Usage Limit)

- **每学员每单元每天 1 次 tutorial session** (`daily_limit: 1` in student_model.json)。这是防止依赖 (dependency) 的硬约束。
- 为什么限频：Oxford tutorial 的价值在于强制学生独立完成 pre-task 和 Socratic 准备。无限频则学生会把 tutorial 当 QA 通道，丧失 retrieval practice 和 struggle 的认知收益 (Vygotsky zone of proximal development + Bjork desirable difficulty)。
- 触发限频时：fellow 输出 'You have used today\'s tutorial quota for this unit. Return tomorrow. In the meantime, re-attempt the drill that triggered your last weak_loop.'

### Exit Artifact (mandatory before leaving tutorial)

Before the fellow signs off, the student must write (in the cell below) the following 3 items. **If any item is missing, the tutorial does not count as completed and does not update student_model.json.**

1. **Two cognitive blind spots identified during this tutorial** (not generic - specific to *your* reasoning gaps, e.g. 'I conflated logit scaling with loss scaling in InfoNCE').
2. **One counterfactual**: if you had to deploy a multimodal ad-creative system *without* CLIP (only BLIP-2), what would break first?
3. **One recommended review unit** from a *different* skill (e.g. skill3-causal-inference, skill2-evaluation) that you believe intersects with today's material. Justify the intersection in one sentence.

```
# Exit artifact (write below):

# 1. Two blind spots:

# 2. Counterfactual (no-CLIP system):

# 3. Cross-skill review unit + intersection justification:

```

### Tutorial sign-off

When all three exit items are filled and the fellow has read them aloud (forced articulation), the student may save student_model.json with updated `socratic_rounds_completed` and `ilo_mastery` levels. The fellow does **not** write the answers for the student. The fellow does **not** round up mastery levels out of kindness. Mastery is earned, not granted.
